# RAG Evaluation with DeepEval

This notebook evaluates a RAG (Retrieval-Augmented Generation) pipeline using the [DeepEval](https://github.com/confident-ai/deepeval) framework.

It uses the file `evaluation_set.json`, which contains for each question:
- `Question`: the user query
- `output_answer`: the answer generated by the RAG pipeline
- `output_chunks`: the chunks generated by the RAG pipeline
- `expected_answer`: ground truth
- `expected_chunks`: expected chunks

We evaluate two things:

1. **Retrieval quality** (how good are the retrieved chunks?)
   - `ContextualPrecisionMetric` - are the most relevant chunks ranked higher?
   - `ContextualRecallMetric` - do the retrieved chunks contain everything needed to produce the expected answer?
   - `ContextualRelevancyMetric` - how relevant, on average, are the retrieved chunks to the question?

2. **Answer quality** (how good is the generated answer?)
   - `FaithfulnessMetric` - is the predicted answer factually consistent with the retrieved context (i.e. not hallucinating)?
   - `AnswerRelevancyMetric` - does the predicted answer actually address the question?
   - `GEval` (correctness) - how close is the predicted answer to the expected ground-truth answer?

BLEU=BP⋅exp(∑n=1,N w_n * log p_n)

Brevity Penalty ( BP ): BP=1 if candidate length c> reference length r ; otherwise BP=e^1−r/c.

N-gram Precision (p_n): Ratio of matching n-grams in the candidate to total n-grams in the candidate, with counts clipped by reference counts.

Weights (w_n): Typically uniform (e.g., 1/N ).

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) measures recall based on overlapping n-grams, longest common subsequences (LCS), or skip bigrams.

The most common variant, ROUGE-N, 

calculates recall as:
ROUGE-N= ∑n-gram∈reference min(Count_candidate, Count_reference) / ∑n-gram∈reference Count(n-gram)


ROUGE-L: Uses the Longest Common Subsequence (LCS) to evaluate sentence-level similarity, often combining precision and recall into an F1 score. 

ROUGE-S: Measures skip-bigram overlap to capture flexible word ordering.

All metrics in DeepEval are backed by an LLM-as-judge. This notebook uses **open-source models hosted on [Groq](https://console.groq.com)** as the judge (via Groq's OpenAI-compatible endpoint), so you'll need a `GROQ_API_KEY` set up before running the evaluation cells.

## 1. Setup

In [1]:
import os

# Choose which LLM judge provider to use: "groq" or "huggingface"
# - Groq: very fast, but the free/dev tier has a small model catalog and tight rate limits,
#   which is limiting when running many (test_case x metric) judge calls.
# - Hugging Face: Inference Providers expose an OpenAI-compatible endpoint giving access to a
#   much larger catalog of open-weight models (Llama, Qwen, DeepSeek, Mistral, etc.), routed
#   across multiple backing providers (novita, together, fireworks, sambanova, ...), which in
#   practice gives more headroom before hitting rate limits.
JUDGE_PROVIDER = "huggingface"  # "groq" or "huggingface"

# Groq: OpenAI-compatible endpoint.
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

# Hugging Face Inference Providers: OpenAI-compatible router endpoint.
# Docs: https://huggingface.co/docs/inference-providers/en/guides/openai-compatibility
HF_BASE_URL = "https://router.huggingface.co/v1"

# Set the relevant key below, or export it as an environment variable before starting Jupyter.
# os.environ["GROQ_API_KEY"] = "gsk_..."
# os.environ["HUGGING_FACE_TOKEN"] = "hf_..."

if JUDGE_PROVIDER == "groq":
    assert os.environ.get("GROQ_API_KEY"), (
        "Please set GROQ_API_KEY (from https://console.groq.com/keys) before running evaluations."
    )
elif JUDGE_PROVIDER == "huggingface":
    assert os.environ.get("HUGGING_FACE_TOKEN"), (
        "Please set HUGGING_FACE_TOKEN (from https://huggingface.co/settings/tokens) before running evaluations. "
        "A free account works; a PRO account or a token with billing enabled unlocks higher rate limits "
        "on Inference Providers."
    )
else:
    raise ValueError(f"Unknown JUDGE_PROVIDER: {JUDGE_PROVIDER!r}")


In [2]:
import json
import pandas as pd
import json
import os 
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    GEval,
)
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import LocalModel  # generic OpenAI-compatible wrapper, used here for Groq

## 2. Load the data

Update `DATA_PATH` if you place the file elsewhere.

In [3]:
# evaluation_set.json is expected in the same directory as this notebook.
DATA_PATH = "../../eval_dataset/evaluation_set.json" # Change path as per your evaluation_set.json

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} evaluation records")
print("Keys:", raw_data[0].keys())


Loaded 1 evaluation records
Keys: dict_keys(['question', 'document_id', 'expected_chunks', 'expected_context', 'expected_answer', 'output_chunks', 'output_answer'])


In [4]:
# Quick look at one record
sample = raw_data[0]

print("Question:", sample["question"])
print("\nExpected Answer:", sample["expected_answer"])
print("\nOutput Answer:", sample["output_answer"][:500], "..." if len(sample["output_answer"]) > 500 else "")
print("\nNumber of expected chunks:", len(sample.get("expected_chunks", [])))
print("Number of output chunks:", len(sample.get("output_chunks", [])))

if sample.get("expected_chunks"):
    print("\nExpected chunk preview:", sample["expected_chunks"][0].get("text", "")[:300], "...")

if sample.get("output_chunks"):
    print("\nOutput chunk preview:", sample["output_chunks"][0].get("text", "")[:300], "...")


Question: What telangana government has done on awareness for innovation?

Expected Answer: The Telangana government organized a “Walk for Innovation” as part of the #ఇంటింటాinnovator Exhibition 2022.  The event, held on July 30, 2022 in Yadadri Bhuvanagiri district, included an awareness session on innovation for the public.  It brought together more than 300 school students and district administration officials to sensitize locals on the importance of innovation.

Output Answer: Based on the provided context, the Telangana government conducted a **Walk for Innovation** and an **Awareness Session on Innovation** for the public in Yadadri Bhuvanagiri district on July 30, 2022. This was organized as part of the #ఇంటింటాinnovator Exhibition 2022 outreach.

*Relevant context snippet:* `"As a part of our #ఇంటింటాinnovator Exhibition 2022 outreach, we conducted Walk for Innovation, Awareness Session on Innovation for the public of Yadadri Bhuvanagiri."` 

Number of expected chunks: 3
Number

## 3. Build DeepEval `LLMTestCase`s

Each record in `evaluation_set.json` has this structure:

```text
question
document_id
expected_chunks
expected_context
expected_answer
output_chunks
output_answer
```

Mapping used here:
- `input` -> `question`
- `actual_output` -> `output_answer`
- `expected_output` -> `expected_answer`
- `retrieval_context` -> text from `output_chunks` (what the retriever actually returned)
- `context` -> text from `expected_chunks` / `expected_context` 


In [5]:
def chunk_text(chunk):
    """Extract chunk text from the evaluation-set chunk format."""
    if isinstance(chunk, str):
        return chunk
    if isinstance(chunk, dict):
        return chunk.get("text", "")
    return str(chunk)


def build_test_case(record: dict) -> LLMTestCase:
    # Actual retrieved context: what the RAG retriever returned.
    output_chunks = record.get("output_chunks", [])
    retrieval_context = [chunk_text(chunk) for chunk in output_chunks]

    # Gold/expected context. Prefer expected_chunks when available because
    # it preserves the individual expected chunks used for retrieval evaluation.
    expected_chunks = record.get("expected_chunks", [])
    expected_context = record.get("expected_context", [])
    
    if expected_chunks:
        gold_context = [chunk_text(chunk) for chunk in expected_chunks]
    elif isinstance(expected_context, list):
        gold_context = [chunk_text(chunk) for chunk in expected_context]
    elif expected_context:
        gold_context = [str(expected_context)]
    else:
        gold_context = []

    return LLMTestCase(
        input=record["question"],
        actual_output=record.get("output_answer", ""),
        expected_output=record.get("expected_answer", ""),
        retrieval_context=retrieval_context,
        context=gold_context,
    )


test_cases = [build_test_case(r) for r in raw_data]
print(f"Built {len(test_cases)} test cases")


Built 1 test cases


## 4. Define metrics

### Retrieval metrics
- **ContextualPrecision**: evaluates whether the retrieved `output_chunks` are relevant and appropriately ranked relative to the expected context.
- **ContextualRecall**: evaluates how well the retrieved `output_chunks` cover the expected information/context.
- **ContextualRelevancy**: evaluates whether the retrieved chunks are relevant to the question.

### Answer metrics
- **Faithfulness**: evaluates whether `output_answer` is supported by the retrieved `output_chunks`.
- **AnswerRelevancy**: evaluates whether `output_answer` addresses the question.
- **Correctness (GEval)**: compares `output_answer` with `expected_answer`.


### Judge model: Groq or Hugging Face (open-source models)

Both providers expose an **OpenAI-compatible** chat-completions endpoint, so we point DeepEval's generic
`LocalModel` wrapper at whichever one is selected via `JUDGE_PROVIDER` above, instead of using OpenAI directly.
This lets every metric run its judging with an open-weight model, without being tied to a single vendor's
rate limits or model catalog.

#### Option A: Groq

Current Groq **production** text models (as of Aug 2026), excluding `qwen/qwen3.6-27b` per requirement:

| Model ID | Notes |
|---|---|
| `openai/gpt-oss-120b` | Best overall quality/reasoning of the open text models on Groq - recommended default judge |
| `llama-3.3-70b-versatile` | Strong general-purpose alternative |
| `openai/gpt-oss-20b` | Smaller/faster, cheaper, slightly less reliable judgments |
| `llama-3.1-8b-instant` | Fastest/cheapest, best for quick smoke tests rather than final scoring |

Groq's rate limits (especially free tier) are small relative to the volume of judge calls a full
DeepEval run makes (each test case x metric can trigger several sub-calls), so long runs can get
throttled.

#### Option B: Hugging Face Inference Providers

Hugging Face routes chat-completion requests to whichever backing provider is hosting a given model
(novita, together, fireworks-ai, sambanova, hyperbolic, ...), so the effective rate limit is generally
much higher, and the model catalog is far larger than Groq's. Model IDs use the `<repo_id>` format, and
you can optionally pin a specific backing provider with `<repo_id>:<provider>`.

| Model ID | Notes |
|---|---|
| `meta-llama/Llama-3.3-70B-Instruct` | Strong general-purpose judge, good default |
| `Qwen/Qwen2.5-72B-Instruct` | Strong alternative, good at following judging rubrics |
| `deepseek-ai/DeepSeek-V3` | Larger MoE model, high quality, slower |
| `meta-llama/Llama-3.1-8B-Instruct` | Fastest/cheapest, best for quick smoke tests |

Pick a provider/model via `JUDGE_PROVIDER` and `GROQ_JUDGE_MODEL` / `HF_JUDGE_MODEL` below. You can also
use two different models - e.g. a strong one for correctness/faithfulness and a fast one for cheaper
metrics - by creating separate `LocalModel` instances.


In [6]:
THRESHOLD = 0.5

# --- Groq model options ---
GROQ_JUDGE_MODEL = "openai/gpt-oss-120b"  # "openai/gpt-oss-20b" "llama-3.3-70b-versatile" "llama-3.1-8b-instant"

# --- Hugging Face model options (served via HF Inference Providers) ---
# Format is "<repo_id>" or "<repo_id>:<provider>" to pin a specific backing provider.
# Leaving off the provider lets HF route to whichever provider is available for that model.
# HF_JUDGE_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
# Other good options:
# HF_JUDGE_MODEL = "Qwen/Qwen2.5-72B-Instruct"
HF_JUDGE_MODEL = "deepseek-ai/DeepSeek-V3"
# HF_JUDGE_MODEL = "meta-llama/Llama-3.1-8B-Instruct:novita"  # faster/cheaper, pinned provider

if JUDGE_PROVIDER == "groq":
    JUDGE_MODEL = GROQ_JUDGE_MODEL
    judge = LocalModel(
        model=JUDGE_MODEL,
        api_key=os.environ["GROQ_API_KEY"],
        base_url=GROQ_BASE_URL,
        temperature=0,
    )
else:  # "huggingface"
    JUDGE_MODEL = HF_JUDGE_MODEL
    judge = LocalModel(
        model=JUDGE_MODEL,
        api_key=os.environ["HUGGING_FACE_TOKEN"],
        base_url=HF_BASE_URL,
        temperature=0,
    )

print(f"Using {JUDGE_PROVIDER} judge model: {JUDGE_MODEL}")

# --- Retrieval-quality metrics ---
contextual_precision = ContextualPrecisionMetric(threshold=THRESHOLD, model=judge, include_reason=True)
contextual_recall = ContextualRecallMetric(threshold=THRESHOLD, model=judge, include_reason=True)
contextual_relevancy = ContextualRelevancyMetric(threshold=THRESHOLD, model=judge, include_reason=True)

# --- Answer-quality metrics ---
faithfulness = FaithfulnessMetric(threshold=THRESHOLD, model=judge, include_reason=True)
answer_relevancy = AnswerRelevancyMetric(threshold=THRESHOLD, model=judge, include_reason=True)

correctness = GEval(
    name="Correctness",
    criteria=(
        "Determine whether the actual output is factually correct and semantically "
        "equivalent to the expected output, given the input question. Minor differences "
        "in wording, structure, or level of detail are fine as long as the core facts match."
    ),
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    threshold=THRESHOLD,
    model=judge,
)

retrieval_metrics = [contextual_precision, contextual_recall, contextual_relevancy]
answer_metrics = [faithfulness, answer_relevancy, correctness]
# all_metrics = retrieval_metrics # + answer_metrics


Using huggingface judge model: deepseek-ai/DeepSeek-V3


## 5. Run the evaluation

`deepeval.evaluate` runs every metric against every test case and returns structured results.
This will make LLM calls for each (test case, metric) pair, so with 10 questions x 6 metrics expect ~60+ judge calls (some metrics like `ContextualPrecision`/`Recall`/`Relevancy` and `Faithfulness` make several sub-calls per test case to score individual chunks/claims, so actual call volume will be higher).

Groq's developer-tier rate limits are generous but finite (see the table above, e.g. 1K RPM for `gpt-oss-120b`) - if you hit 429s on a larger dataset, either switch `JUDGE_MODEL` to a higher-limit model, add `async_config`/concurrency limits, or run in smaller batches.

## Evaluate quality of retrieved chunks

In [17]:
JUDGE_PROVIDER = "groq"
eval_chunks_results1 = evaluate(
    test_cases=list(test_cases),
    metrics=retrieval_metrics[:2],
)

✨ You're running DeepEval's latest Contextual Precision Metric! (using None (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using None (Local Model), strict=False, 
async_mode=True)...



Metrics Summary

  - ✅ Contextual Precision (score: 0.8055555555555555, threshold: 0.5, strict: False, evaluation model: None (Local Model), reason: The score is 0.81 because the first node in retrieval contexts is relevant, stating 'As a part of our #ఇంటింటాinnovator Exhibition 2022 outreach, we conducted and Awareness Session on Innovation,' which aligns with the input. However, the second node is irrelevant, discussing local recognition for innovators without mentioning awareness sessions. The third and fourth nodes are relevant, confirming the 'Walk for Innovation' and 'Awareness Session on Innovation,' but the fifth node is irrelevant, listing events without specific awareness details. The irrelevant nodes should be ranked lower to improve precision., error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.5, strict: False, evaluation model: None (Local Model), reason: The score is 1.00 because all sentences in the expected output are fully supported by the 3rd node in th

✓ Evaluation completed 🎉! (time taken: 21.18s | token cost: 0.0 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [18]:
eval_chunks_results = evaluate(
    test_cases=list(test_cases),
    metrics=retrieval_metrics[2:],
)

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using None (Local Model), strict=False, 
async_mode=True)...



Metrics Summary

  - ❌ Contextual Relevancy (score: 0.34615384615384615, threshold: 0.5, strict: False, evaluation model: None (Local Model), reason: The score is 0.35 because the retrieval context contains numerous irrelevant details such as dates, locations, and contact information ('August 7, 2022', 'Location: Balaji Institute of Technology & Science', 'E-Mail: support-startup@telangana.gov.in'), which do not address the Telangana government's efforts on awareness for innovation. However, a few relevant statements like 'As a part of our #ఇంటింటాinnovator Exhibition 2022 outreach, we conducted an Awareness Session on Innovation' highlight specific initiatives, but these are overshadowed by the majority of irrelevant content., error: None)

For test case:

  - input: What telangana government has done on awareness for innovation?
  - actual output: Based on the provided context, the Telangana government conducted a **Walk for Innovation** and an **Awareness Session on Innovation** f

✓ Evaluation completed 🎉! (time taken: 41.55s | token cost: 0.0 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

**ContextualPrecisionMetric** -
Retriever is finding the relevant chunks, but ranking them too low.

The problem is not that the relevant information is missing.
The problem is: The relevant information appears too late in the ranked results.

**Precision vs Contextual Precision**

This distinction is important for your RAG cohort.

Normal retrieval precision

Suppose you retrieve 5 chunks:

Relevant
Relevant
Irrelevant
Irrelevant
Irrelevant

You have:

Precision = 2 / 5 = 0.40

But contextual precision goes one step further.

It considers the ranking position.

Compare:

Case A
1. Relevant
2. Relevant
3. Irrelevant
4. Irrelevant
5. Irrelevant

Good.

Case B
1. Irrelevant
2. Irrelevant
3. Relevant
4. Relevant
5. Irrelevant

Worse.

Both have same precision

2 relevant / 5 = 0.40

But Case A has much better contextual precision because the relevant information is retrieved earlier.

**What can be done?**

Reranking

Improve chunking - If your chunks are too large or poorly structured, embeddings can represent the general topic rather than the specific event.

Check embedding model if it is able to understand the difference between general topics and specific events. Embedding provides the properties for the content.


**Contextual Recall** measures whether the retrieved context contains the information needed to answer the expected answer.

In simple terms:

"Did my retriever bring back enough of the right information?"

It is mainly about retrieval completeness.

Expected Answer

├── Sentence 1 → ✅ Found in retrieved context

├── Sentence 2 → ✅ Found in retrieved context

└── Sentence 3 → ❌ Not found

**What can be done?**

Reranking will not help because in recall the chunk is not retrieved.

Data is available in the DB? If not then it is clearly an ingestion problem.

If available then why it is not picked? Experiment with:

embeddings - Check embedding model if it is able to understand the difference between general topics and specific events. Embedding provides the properties for the content.

chunking - Chunk size. If the information is very specific, is it gets burried in a large chunk.

hybrid search - Filter irrelevant content with BM25 indexes straight and then look for vector indexes.

top-K - Increase top-k. 

search parameters - If HNSW increase ef number or nproble in case of cluster index.

query expansion 




**Contextual Relevancy**

How relevant is the retrieved context to the user's group?

If score is less - Retrieved context contains a lot of information, but much of it isn't useful for answering the specific question.

It looks at the retrieved chunks and checks whether the information in them is actually useful for answering the query

**What can be done**
Check chunk size. If the chunk size is quite large then it might get lot of irrelevant 

semantic/paragraph chunking

+

heading-aware chunking

+

reasonable chunk size

+

small overlap

Hybrid retrieval - BM25 + Dense search

Reranking


**Difference between Contextual Precision and Contextual Recall and Contextual Relevancy**

Contextual Precision	Did I retrieve the right chunks near the top?

Contextual Recall	Did I retrieve all/most of the information needed?

Contextual Relevancy - Looks at: Is the retrieved content relevant to the query at all?

If recall is higher say 3 out of 5 are expected chunks and we get most of the expected or complete answer but relevancy will be lower as all the chunks are not contributing for the answer.


In [ ]:
JUDGE_PROVIDER = "groq"
eval_chunks_results = evaluate(
    test_cases=list(test_cases),
    metrics=answer_metrics[:2],
)

# 2*0.2 = 0.4 - score has no relationship with the threshold.
# 2*0.1 = 0.2 

✨ You're running DeepEval's latest Faithfulness Metric! (using None (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using None (Local Model), strict=False, 
async_mode=True)...

/home/neeraj/Projects/pdf_to_markdown/pdf_to_md_env/lib/python3.12/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.5, strict: False, evaluation model: None (Local Model), reason: The score is 1.00 because there are no contradictions found, indicating a perfect alignment between the actual output and the retrieval context, which is absolutely fantastic!, error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: None (Local Model), reason: The score is 1.00 because the output perfectly addresses the input question about Telangana government's efforts on awareness for innovation, with no irrelevant statements found., error: None)

For test case:

  - input: What telangana government has done on awareness for innovation?
  - actual output: Based on the provided context, the Telangana government conducted a **Walk for Innovation** and an **Awareness Session on Innovation** for the public in Yadadri Bhuvanagiri district on July 30, 2022. This was organized as part of the #ఇంటింటాinnovator Exhibition 20

✓ Evaluation completed 🎉! (time taken: 8.69s | token cost: 0.0 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [9]:
JUDGE_PROVIDER = "huggingface"
eval_chunks_results = evaluate(
    test_cases=list(test_cases),
    metrics=answer_metrics[2:],
)

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using None (Local Model), strict=False, 
async_mode=True)...

/home/neeraj/Projects/pdf_to_markdown/pdf_to_md_env/lib/python3.12/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.8, threshold: 0.5, strict: False, evaluation model: None (Local Model), reason: The actual output demonstrates a strong understanding of the context, correctly identifying the Telangana government's initiatives for innovation awareness, including the 'Walk for Innovation' and an 'Awareness Session on Innovation'. Although it lacks specific details about the number of participants and the goal of sensitizing locals, it shows semantic equivalence with the expected output by conveying the core facts about the event, such as its date, location, and purpose., error: None)

For test case:

  - input: What telangana government has done on awareness for innovation?
  - actual output: Based on the provided context, the Telangana government conducted a **Walk for Innovation** and an **Awareness Session on Innovation** for the public in Yadadri Bhuvanagiri district on July 30, 2022. This was organized as part of the #ఇంటింటాinnovator Exhibiti

✓ Evaluation completed 🎉! (time taken: 13.92s | token cost: 0.0 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

**Faithfulness** is one of the most useful RAG evaluation metrics for **detecting hallucination**, especially when your system is expected to answer only from the retrieved context.

SCore 1.0 - Means the evaluator found that the claims made in the LLM's answer are supported by the retrieved context.

Think of it as:

"**Did the answer stay faithful to the information that was retrieved?**"

In a RAG system, you can think about hallucination like this:

**User Query → Retrieval → Context → LLM Answer**

Faithfulness primarily evaluates:

Context → Answer

It asks:

"Did the LLM make claims that are unsupported by the retrieved context?"

So a low faithfulness score is a strong signal of potential hallucination.

Example

Suppose your retrieved context says:

Milvus supports HNSW and IVF indexes for approximate nearest-neighbor search.

The LLM generates:

Milvus supports HNSW and IVF indexes. HNSW generally provides faster search while IVF requires less memory.

The first statement is supported.

But if the retrieved context doesn't say anything about memory usage, the second statement may be considered unsupported.

So:

**Faithfulness ↓ → possibility of hallucination ↑**

**Faithfulness ≠ factual correctness**

This is very important when interpreting your evaluation results. If the context itself is wrong then the generated answer will also be wrong.


**Relevancy**

**How much the Answer is relevant to question?**

The answer can be relevant, but if not supported by the retrieved context then the **relevancy might be higher but failthfulness would be lower. Very high chances of hallucination.**

| Answer Relevancy | Faithfulness | Interpretation |
|---|---|---|
| **High** | **High** | ✅ Relevant and grounded |
| **High** | **Low** | ⚠️ **Relevant but potentially hallucinated** |
| **Low** | **High** | ⚠️ Grounded but doesn't answer the question |
| **Low** | **Low** | ❌ Bad answer + possible hallucination |

=======================

Question: What happens under Condition A?

Reality: Condition A → Exception X

Training data: Mostly Condition B → Outcome Y

LLM: Answers with Outcome Y.

Think of it as three different failure modes:

| Situation	| What happened?	| Likely problem |
|---|---|---|
| Model answers B because B is much more common in training data	| Model's learned prior dominates	| Training-data bias / learned prior |
| Context explicitly says A → X, but model answers B → Y	| Model ignores/contradicts retrieved evidence	| Hallucination / grounding failure |
| Context doesn't contain information about A, and model fills the gap with B	| Model relies on prior knowledge	| Unsupported generation / hallucination |


**GEval (Generative Evaluation)** is a framework where an LLM acts as an evaluator/judge and scores another LLM's output against criteria you define.

So the judge considers the answer substantially correct.

The important part is the reason:

"The actual output demonstrates a strong understanding of the context, correctly identifying the Telangana government's initiatives..."

But:

"Although it lacks specific details about the goal of sensitizing locals and student numbers ..."

Is the generated answer correct compared with the expected answer, given the question?

But if context does not have information about senitizing locals but answer provides information on it then Faithfulness would be lower and GEval Correctness would be higher.

In [17]:
eval_res = [eval_chunks_results, eval_chunks_results1]

## 6. Collect results into a DataFrame

Flatten the per-test-case, per-metric results into a tidy table for analysis/export.

In [18]:
rows = []
for eval_test_result in eval_res:
    for i, tc_result in enumerate(eval_test_result.test_results):
        original = raw_data[i]
        row = {
            "question": tc_result.input,
            "document_id": original.get("document_id"),
            "expected_answer": tc_result.expected_output,
            "output_answer": tc_result.actual_output,
            "expected_chunk_count": len(original.get("expected_chunks", [])),
            "output_chunk_count": len(original.get("output_chunks", [])),
        }
        for metric_data in tc_result.metrics_data:
            row[f"{metric_data.name}_score"] = metric_data.score
            row[f"{metric_data.name}_success"] = metric_data.success
            row[f"{metric_data.name}_reason"] = metric_data.reason
        rows.append(row)

results_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
results_df.head()


,question,document_id,expected_answer,output_answer,expected_chunk_count,output_chunk_count,Correctness [GEval]_score,Correctness [GEval]_success,Correctness [GEval]_reason,Contextual Precision_score,Contextual Precision_success,Contextual Precision_reason,Contextual Recall_score,Contextual Recall_success,Contextual Recall_reason
0,What telangana government has done on awareness for innovation?,91dffa0d-9183-447d-b11f-aaca1e7afb50,The Telangana government organized a “Walk for Innovation” as part of the #ఇంటింటాinnovator Exhibition 2022. The ev...,"Based on the provided context, the Telangana government conducted a **Walk for Innovation** and an **Awareness Sessi...",3,5,0.8,True,"The actual output demonstrates a strong understanding of the context, correctly identifying the Telangana government...",NaN,NaN,NaN,NaN,NaN,NaN
1,What telangana government has done on awareness for innovation?,91dffa0d-9183-447d-b11f-aaca1e7afb50,The Telangana government organized a “Walk for Innovation” as part of the #ఇంటింటాinnovator Exhibition 2022. The ev...,"Based on the provided context, the Telangana government conducted a **Walk for Innovation** and an **Awareness Sessi...",3,5,NaN,NaN,NaN,0.416667,False,"The score is 0.42 because irrelevant nodes are ranked higher than relevant ones. For example, node 1 discusses an aw...",0.666667,True,The score is 0.67 because sentences 1 and 2 in the expected output are well-supported by the 3rd node in the retriev...


In [28]:
results_df.to_csv("../../eval_output/results.csv")

## 7. Summary scores

Average score per metric across all questions, plus pass rate (fraction of questions where score >= threshold).

In [19]:
score_cols = [c for c in results_df.columns if c.endswith("_score")]
success_cols = [c for c in results_df.columns if c.endswith("_success")]

summary = pd.DataFrame({
    "metric": [c.replace("_score", "") for c in score_cols],
    "avg_score": [results_df[c].mean() for c in score_cols],
    "pass_rate": [results_df[sc].mean() for sc in success_cols],
})
summary

,metric,avg_score,pass_rate
0,Correctness [GEval],0.800000,1.0
1,Contextual Precision,0.416667,0.0
2,Contextual Recall,0.666667,1.0


In [20]:
print("=== Retrieval metrics (chunk quality) ===")
display(summary[summary["metric"].str.contains("Contextual")])

print("\n=== Answer metrics (generation quality) ===")
display(summary[~summary["metric"].str.contains("Contextual")])

=== Retrieval metrics (chunk quality) ===


,metric,avg_score,pass_rate
1,Contextual Precision,0.416667,0.0
2,Contextual Recall,0.666667,1.0



=== Answer metrics (generation quality) ===


,metric,avg_score,pass_rate
0,Correctness [GEval],0.8,1.0


## 8. Inspect low-scoring examples

Useful for debugging: find questions where retrieval or generation quality was weak.

In [21]:
def show_low_scores(df: pd.DataFrame, metric: str, n: int = 3):
    col = f"{metric}_score"
    reason_col = f"{metric}_reason"
    worst = df.sort_values(col).head(n)
    for _, r in worst.iterrows():
        print(f"Question: {r['question']}")
        print(f"{metric} score: {r[col]:.2f}")
        print(f"Reason: {r[reason_col]}")
        print("-" * 80)

# Example: worst 3 by Faithfulness
show_low_scores(results_df, "Faithfulness", n=3)


KeyError: 'Faithfulness_score'

In [22]:
# Example: worst 3 by Contextual Recall
show_low_scores(results_df, "Contextual Recall", n=3)

Question: What telangana government has done on awareness for innovation?
Contextual Recall score: 0.67
Reason: The score is 0.67 because sentences 1 and 2 in the expected output are well-supported by the 3rd node in the retrieval context, which details the event's name, date, location, and purpose. However, sentence 3 mentions 'more than 300 school students and district administration officials,' a detail not found in any of the retrieval context nodes.
--------------------------------------------------------------------------------
Question: What telangana government has done on awareness for innovation?
Contextual Recall score: nan
Reason: nan
--------------------------------------------------------------------------------


## 9. Non-LLM-judge metrics (statistical / small-model based)

Everything above (`ContextualPrecisionMetric`, `ContextualRecallMetric`, `ContextualRelevancyMetric`,
`FaithfulnessMetric`, `AnswerRelevancyMetric`, and `GEval`/Correctness) uses **LLM-as-a-judge**: an LLM
reads the test case and outputs a score + reason. That's flexible and semantically aware, but it costs
an API call (or several, since these metrics score claim-by-claim/chunk-by-chunk) per test case per metric.

DeepEval also ships a `deepeval.scorer.Scorer` class with classic **statistical NLP metrics** (ROUGE, BLEU,
exact match, BERTScore) and a couple of **small, purpose-built classifier models** (SummaC for faithfulness,
a cross-encoder/bi-encoder for answer relevancy) that don't call a generative LLM at all. Below, each of the
five non-`GEval` metrics is reimplemented as a custom `BaseMetric` on top of `Scorer`, so they can be run
through the same `evaluate()` call.

**How each one is actually computed internally:**

| Metric | Non-LLM technique | How it works |
|---|---|---|
| **Contextual Recall** | ROUGE-L (n-gram/LCS overlap) | For each *expected/gold* chunk, take the best ROUGE-L F1 against every *retrieved* chunk, then average across gold chunks. ROUGE-L finds the Longest Common Subsequence of words between two texts and turns it into a precision/recall/F1 triple — it rewards retrieved chunks that reuse the same wording as the gold chunk, in the same relative order. |
| **Contextual Precision** | ROUGE-L + Average Precision (IR ranking metric) | Each *retrieved* chunk is labeled "relevant" if its best ROUGE-L F1 against any gold chunk clears a threshold. Average Precision is then computed over that ranked relevance list, so chunks that are both relevant *and* ranked earlier by the retriever count for more — this mirrors what an LLM judge is approximating, but with an n-gram match instead of a semantic one. |
| **Contextual Relevancy** | BERTScore (contextual embeddings) | ROUGE/BLEU are n-gram overlap metrics, which is a poor fit for question-vs-passage relevance (a question and a relevant passage rarely share literal n-grams). Instead this uses BERTScore: both texts are embedded token-by-token with a pretrained transformer (`microsoft/deberta-large-mnli`), each question token is greedily matched to its most similar chunk token by cosine similarity, and those similarities are aggregated into precision/recall/F1. It's still not generative — no text is produced, only a similarity score. |
| **Faithfulness** | SummaC (zero-shot NLI) | `Scorer.faithfulness_score` runs SummaCZS, which splits the retrieved context and the generated answer into sentences and scores every (context-sentence, answer-sentence) pair with a small pretrained **Natural Language Inference (NLI)** model (predicting entailment / neutral / contradiction), then aggregates the entailment probabilities. This is conceptually the same "is the answer supported by the context" question the LLM judge asks, just answered by a lightweight classifier instead of a generative model. |
| **Answer Relevancy** | Cross-encoder / bi-encoder semantic similarity | `Scorer.answer_relevancy_score` feeds the question and the generated answer into a sentence-transformers **cross-encoder** (jointly, for higher accuracy) or a **bi-encoder** (embed each separately, then cosine/dot-product similarity, faster). Either way it's a single small transformer producing a similarity score — no generation, no reasoning trace. |

BLEU (`Scorer.sentence_bleu_score`) is also available if you want a stricter, precision-+ brevity-penalty
based alternative to ROUGE for any of the text-overlap comparisons above — it tends to be harsher than
ROUGE on paraphrased text, since it does not use stemming and weighs exact n-gram precision more heavily.


In [8]:
# One-time installs for the non-LLM scorers used below.
# - rouge-score: ROUGE-1/2/L
# - nltk: BLEU + tokenization
# - bert-score: BERTScore (contextual-embedding similarity)
# - sentence-transformers: cross-encoder / bi-encoder used by Scorer.answer_relevancy_score
# - summac: SummaCZS NLI-based faithfulness/consistency model
# - torch: backend for bert-score / sentence-transformers / summac
%pip install --quiet rouge-score nltk bert-score sentence-transformers summac torch

import nltk
for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Could not download nltk resource {pkg!r}: {e}")


Note: you may need to restart the kernel to use updated packages.


### Custom non-LLM `BaseMetric` classes

Each class below follows DeepEval's `BaseMetric` interface (`measure`, `a_measure`, `is_successful`, `score`,
`reason`) so they slot into `evaluate()` exactly like the LLM-judge metrics did. `async_mode = False` marks
them as synchronous, local computations - no network/API calls are made.


In [13]:
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase
from deepeval.scorer import Scorer


class ROUGEContextualRecallMetric(BaseMetric):
    """Non-LLM Contextual Recall: best ROUGE-L F1 of each gold chunk against the
    retrieved chunks, averaged. See table above for the full explanation."""

    def __init__(self, threshold: float = 0.5, rouge_type: str = "rougeL"):
        self.threshold = threshold
        self.rouge_type = rouge_type
        self.async_mode = False

    def measure(self, test_case: LLMTestCase) -> float:
        expected_chunks = test_case.context or []
        output_chunks = test_case.retrieval_context or []
        if not expected_chunks or not output_chunks:
            self.score, self.success = 0.0, False
            self.reason = "Missing expected_chunks (context) or output_chunks (retrieval_context)."
            return self.score

        per_chunk_best = [
            max(
                Scorer.rouge_score(target=exp, prediction=out, score_type=self.rouge_type)
                for out in output_chunks
            )
            for exp in expected_chunks
        ]
        self.score = sum(per_chunk_best) / len(per_chunk_best)
        self.success = self.score >= self.threshold
        matched = sum(1 for s in per_chunk_best if s >= self.threshold)
        self.reason = (
            f"{matched}/{len(per_chunk_best)} expected chunks were matched "
            f"(best {self.rouge_type} F1 >= {self.threshold}) by some retrieved chunk."
        )
        return self.score

    async def a_measure(self, test_case: LLMTestCase, *args, **kwargs) -> float:
        return self.measure(test_case)

    def is_successful(self) -> bool:
        return self.success

    @property
    def __name__(self):
        return "ROUGE Contextual Recall"


class ROUGEContextualPrecisionMetric(BaseMetric):
    """Non-LLM Contextual Precision: rank each retrieved chunk as relevant/not
    (best ROUGE-L F1 against any gold chunk >= relevance_threshold), then compute
    Average Precision over that ranked relevance list."""

    def __init__(self, threshold: float = 0.5, rouge_type: str = "rougeL", relevance_threshold: float = 0.5):
        self.threshold = threshold
        self.rouge_type = rouge_type
        self.relevance_threshold = relevance_threshold
        self.async_mode = False

    def measure(self, test_case: LLMTestCase) -> float:
        expected_chunks = test_case.context or []
        output_chunks = test_case.retrieval_context or []
        if not expected_chunks or not output_chunks:
            self.score, self.success = 0.0, False
            self.reason = "Missing expected_chunks (context) or output_chunks (retrieval_context)."
            return self.score

        relevance = []
        for out in output_chunks:
            best = max(
                Scorer.rouge_score(target=exp, prediction=out, score_type=self.rouge_type)
                for exp in expected_chunks
            )
            relevance.append(1 if best >= self.relevance_threshold else 0)

        hits, precisions = 0, []
        for rank, rel in enumerate(relevance, start=1):
            if rel:
                hits += 1
                precisions.append(hits / rank)

        self.score = (sum(precisions) / hits) if hits else 0.0
        self.success = self.score >= self.threshold
        self.reason = (
            f"{hits}/{len(output_chunks)} retrieved chunks matched a gold chunk "
            f"(best {self.rouge_type} F1 >= {self.relevance_threshold}); "
            f"Average Precision = {self.score:.2f}."
        )
        return self.score

    async def a_measure(self, test_case: LLMTestCase, *args, **kwargs) -> float:
        return self.measure(test_case)

    def is_successful(self) -> bool:
        return self.success

    @property
    def __name__(self):
        return "ROUGE Contextual Precision"


class BERTScoreContextualRelevancyMetric(BaseMetric):
    def __init__(self, threshold=0.5, bert_model="microsoft/deberta-large-mnli"):
        super().__init__()
        self.threshold = threshold
        self.bert_model = bert_model
        self.async_mode = False
        self._scorer = None  # lazy-loaded, built once

    def _get_scorer(self):
        if self._scorer is None:
            from bert_score import BERTScorer
            self._scorer = BERTScorer(model_type=self.bert_model, lang="en", rescale_with_baseline=True)
        return self._scorer

    def measure(self, test_case):
        output_chunks = test_case.retrieval_context or []
        scorer = self._get_scorer()
        f1s = []
        for chunk in output_chunks:
            _, _, f1 = scorer.score(cands=[chunk], refs=[test_case.input])
            f1s.append(f1.item())
        self.score = sum(f1s) / len(f1s) if f1s else 0.0
        self.success = self.score >= self.threshold
        self.reason = f"Average BERTScore-F1 across {len(output_chunks)} chunks is {self.score:.2f}."
        return self.score

    async def a_measure(self, test_case: LLMTestCase, *args, **kwargs) -> float:
        return self.measure(test_case)

    def is_successful(self) -> bool:
        return self.success

    @property
    def __name__(self):
        return "BERTScore Contextual Relevancy"


class SummaCFaithfulnessMetric(BaseMetric):
    """Non-LLM Faithfulness: SummaCZS zero-shot NLI consistency score between the
    retrieved context (source) and the generated answer (prediction)."""

    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold
        self.async_mode = False

    def measure(self, test_case: LLMTestCase) -> float:
        output_chunks = test_case.retrieval_context or []
        source = "\n".join(output_chunks)
        self.score = Scorer.faithfulness_score(target=source, prediction=test_case.actual_output)
        self.success = self.score >= self.threshold
        self.reason = (
            "SummaCZS (sentence-level NLI entailment, aggregated) score between the "
            f"retrieved context and the generated answer is {self.score:.2f}."
        )
        return self.score

    async def a_measure(self, test_case: LLMTestCase, *args, **kwargs) -> float:
        return self.measure(test_case)

    def is_successful(self) -> bool:
        return self.success

    @property
    def __name__(self):
        return "SummaC Faithfulness"


class CrossEncoderAnswerRelevancyMetric(BaseMetric):
    """Non-LLM Answer Relevancy: cross-encoder semantic-similarity score between
    the question and the generated answer."""

    def __init__(self, threshold: float = 0.5, model_name: str = None):
        self.threshold = threshold
        self.model_name = model_name
        self.async_mode = False

    def measure(self, test_case: LLMTestCase) -> float:
        self.score = Scorer.answer_relevancy_score(
            predictions=test_case.actual_output,
            target=test_case.input,
            model_type="cross_encoder",
            model_name=self.model_name,
        )
        self.success = self.score >= self.threshold
        self.reason = (
            "Cross-encoder semantic-similarity score between the question and the "
            f"generated answer is {self.score:.2f}."
        )
        return self.score

    async def a_measure(self, test_case: LLMTestCase, *args, **kwargs) -> float:
        return self.measure(test_case)

    def is_successful(self) -> bool:
        return self.success

    @property
    def __name__(self):
        return "Cross-Encoder Answer Relevancy"


### Instantiate and run the non-LLM metrics

Same `THRESHOLD` and same `test_cases` as before - only the scoring mechanism changes.


In [14]:
rouge_contextual_recall = ROUGEContextualRecallMetric(threshold=THRESHOLD)
rouge_contextual_precision = ROUGEContextualPrecisionMetric(threshold=THRESHOLD)
bertscore_contextual_relevancy = BERTScoreContextualRelevancyMetric(threshold=THRESHOLD)
summac_faithfulness = SummaCFaithfulnessMetric(threshold=THRESHOLD)
cross_encoder_answer_relevancy = CrossEncoderAnswerRelevancyMetric(threshold=THRESHOLD)

non_llm_retrieval_metrics = [rouge_contextual_precision, rouge_contextual_recall, bertscore_contextual_relevancy]
non_llm_answer_metrics = [summac_faithfulness, cross_encoder_answer_relevancy]
# Note: Correctness (GEval) has no non-LLM equivalent here and still needs `judge` from cell 11.


In [15]:
for m in non_llm_retrieval_metrics:
    print("Testing:", m.__name__)
    try:
        evaluate(test_cases=[test_cases[0]], metrics=[m])
    except RecursionError:
        import traceback; traceback.print_exc()
        print(">>> RECURSION IN:", m.__name__)

Testing: ROUGE Contextual Precision


✨ You're running DeepEval's latest ROUGE Contextual Precision Metric! (using None, strict=False, 
async_mode=True)...



Metrics Summary

  - ✅ ROUGE Contextual Precision (score: 1.0, threshold: 0.5, strict: False, evaluation model: None, reason: 4/5 retrieved chunks matched a gold chunk (best rougeL F1 >= 0.5); Average Precision = 1.00., error: None)

For test case:

  - input: What telangana government has done on awareness for innovation?
  - actual output: Based on the provided context, the Telangana government conducted a **Walk for Innovation** and an **Awareness Session on Innovation** for the public in Yadadri Bhuvanagiri district on July 30, 2022. This was organized as part of the #ఇంటింటాinnovator Exhibition 2022 outreach.

*Relevant context snippet:* `"As a part of our #ఇంటింటాinnovator Exhibition 2022 outreach, we conducted Walk for Innovation, Awareness Session on Innovation for the public of Yadadri Bhuvanagiri."`
  - expected output: The Telangana government organized a “Walk for Innovation” as part of the #ఇంటింటాinnovator Exhibition 2022.  The event, held on July 30, 2022 in Yadadri Bh

✓ Evaluation completed 🎉! (time taken: 0.97s | token cost: None USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Testing: ROUGE Contextual Recall


...

Testing: BERTScore Contextual Relevancy


✨ You're running DeepEval's latest BERTScore Contextual Relevancy Metric! (using None, strict=False, 
async_mode=True)...

Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]
Downloading tokenizer_

>>> RECURSION IN: BERTScore Contextual Relevancy


Traceback (most recent call last):
  File "/home/neeraj/Projects/pdf_to_markdown/pdf_to_md_env/lib/python3.12/site-packages/IPython/core/formatters.py", line 282, in catch_format_error
    r = method(self, *args, **kwargs)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/neeraj/Projects/pdf_to_markdown/pdf_to_md_env/lib/python3.12/site-packages/IPython/core/formatters.py", line 972, in __call__
    if self.enabled:
       ^^^^^^^^^^^^
  File "/home/neeraj/Projects/pdf_to_markdown/pdf_to_md_env/lib/python3.12/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.get(obj, cls))  # the G should encode the Optional
                     ^^^^^^^^^^^^^^^^^^
  File "/home/neeraj/Projects/pdf_to_markdown/pdf_to_md_env/lib/python3.12/site-packages/traitlets/traitlets.py", line 666, in get
    return t.cast(G, value)
           ^^^^^^^^^^^^^^^^
RecursionError: maximum recursion depth exceeded

During handling of the above exception, another exception occurred:



In [12]:
# non_llm_chunks_results = evaluate(
#     test_cases=list(test_cases),
#     metrics=non_llm_retrieval_metrics,
# )


In [ ]:
non_llm_answer_results = evaluate(
    test_cases=list(test_cases),
    metrics=non_llm_answer_metrics,
)


## 10. Export results

In [ ]:
results_df.to_csv("deepeval_results_detailed.csv", index=False)
summary.to_csv("deepeval_results_summary.csv", index=False)

# Also save the evaluation configuration for reproducibility.
evaluation_config = {
    "data_file": "evaluation_set.json",
    "judge_provider": JUDGE_PROVIDER,
    "judge_model": JUDGE_MODEL,
    "threshold": THRESHOLD,
    "retrieval_metrics": [m.name for m in retrieval_metrics],
    "answer_metrics": [m.name for m in answer_metrics],
}
with open("deepeval_evaluation_config.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_config, f, indent=2)

print("Saved: deepeval_results_detailed.csv, deepeval_results_summary.csv, deepeval_evaluation_config.json")


## Notes / customization tips

- `evaluation_set.json` must be in the same directory as this notebook.
- `output_chunks` are treated as the actual retrieved context.
- `expected_chunks` are treated as the gold/expected context.
- `output_answer` is the answer generated by your RAG pipeline.
- `expected_answer` is the reference answer.
- `Faithfulness` checks the generated answer against the actual retrieved `output_chunks`.
- Retrieval metrics compare the actual retrieved context with the expected/gold context.
- Start with one metric while debugging, then enable the others as needed.
- Switch judge providers anytime by changing `JUDGE_PROVIDER` in cell 2/11 - no other code needs
  to change, since both Groq and Hugging Face are used through the same OpenAI-compatible
  `LocalModel` wrapper.
- `openai/gpt-oss-120b` (Groq) and `meta-llama/Llama-3.3-70B-Instruct` (Hugging Face) are used as
  the default judges here; smaller models are listed above as lower-cost/faster alternatives.
- If you hit Hugging Face rate limits, try pinning a specific backing provider with
  `"<repo_id>:<provider>"` (e.g. `:together`, `:novita`) instead of letting HF auto-route.
